# floorplanlab — a tour in 10 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datarefinerylab/floorplanlab/blob/main/floorplanlab/tutorial.ipynb)

Generate architectural floor plans with diffusion models, compare them, and build your own
datasets — without wiring a single path by hand.

## How to read this notebook

Every section is marked with what it needs:

| | |
|---|---|
| 🟢 **Runs anywhere** | Works on any machine, no GPU and no model files. Most of this notebook. |
| 🔵 **Needs a GPU + model files** | Actual layout generation. Skip these if you are just browsing. |

So you can run this end to end right now, get real output from most of it, and come back to the
blue sections once you have a GPU and the pre-trained weights.

## Setup 🟢

One install. Nothing else to configure.

In [ ]:
%pip install -q git+https://github.com/datarefinerylab/floorplanlab.git

In [ ]:
import floorplanlab as fpl

print("floorplanlab", fpl.__version__)

---

## 1. What can I generate with? 🟢

`floorplanlab` ships a registry of models. Each entry knows its dataset, its weights, its
sampling script, and *what it is able to do*.

In [ ]:
fpl.available()

Every difference between models is **data, not code** — so you can inspect a model
before running anything.

In [ ]:
spec = fpl.get("ohdw-oesd")

print("label      :", spec.label)
print("dataset    :", spec.dataset)
print("checkpoint :", spec.checkpoint)
print("target sets:", spec.target_sets)
print("scripts    :", spec.scripts)

---

## 2. Generate floor plans 🔵

This is the part that needs a GPU and the pre-trained weights.

`drive_folder` is the folder in your Google Drive holding the weights, patched model sources and
datasets. Name it whatever you like and pass that name.

In [ ]:
lab = fpl.Lab(drive_folder="my-floorplan-files")
lab.setup("ohd-rplan")          # clone repos, install deps, place weights — idempotent

run = lab.generate(target_set=6, num_samples=64)

Metrics come back as **data**, not as something to scroll for:

```
GED 2.100 +/- 0.100   OA 0.9227 +/- 0.0100   (5 rounds)
```

- **GED** — how far the graph rebuilt from the generated layout is from the input. Lower is better.
- **OA** — how well generated spaces face the requested orientations. Higher is better, range 0–1.

In [ ]:
run.metrics.as_dict()
# {'ged': 2.100, 'ged_std': 0.100, 'oa': 0.9227, 'oa_std': 0.0100, 'rounds': 5}

run.show(6)                     # ground truth beside the generated layouts
run.save_to("results/")         # zipped, so it is quick even onto mounted Drive

---

## 3. Compare two models 🔵

Switching models is one string. No runtime restart, because sampling runs as a subprocess and
picks up the newly copied sources.

In [ ]:
lab.setup("hd-rplan", install=False)     # install=False: deps are already there
baseline = lab.generate(target_set=6)

lab.table()

```
run                                              GED                   OA
--------------------------------------------------------------------------
Oriented-HouseDiffusion (O-RPLAN) / t  2.100 +/- 0.100    0.9227 +/- 0.0100
HouseDiffusion (RPLAN) / target 6      3.378 +/- 0.100    0.6181 +/- 0.0100
```

The orientation-aware model scores far better on OA — as it should, since plain HouseDiffusion
never sees orientation as an input at all.

One call turns both runs into a shareable PDF whose **first page is the metrics table**, so the
numbers travel with the pictures:

In [ ]:
lab.report("comparison.pdf")

> ⚠️ **A trap worth knowing.** GED is *not* normalised by graph size — layouts with more rooms
> have more edges and score higher almost mechanically. Compare GED between **models at the same
> target size**, never between target sizes. `lab.table()` prints this warning whenever you mix
> sizes.

---

## 4. Read metrics without a GPU 🟢

The metrics parser is just a function. Here it is on the **real log of an actual sampling run**,
shipped with the library — so you can see exactly what gets extracted.

In [ ]:
from floorplanlab.examples import sample_log

log = sample_log()
print(log[:220], "\n...\n")
print(log.strip().splitlines()[-2])

In [ ]:
m = fpl.parse_metrics(log)

print(m)
print()
print("as data:", m.as_dict())

Note the vocabulary mismatch this quietly fixes: the sampling script prints
`Compatibility`, while the papers and tutorials say `GED`. Both names work.

In [ ]:
print(m.ged, "==", m.compatibility)

# an interrupted run has no summary line, but its per-round values are still usable
partial = "sampling complete\nCompatibility: 4.0\nOA: 0.9\nCompatibility: 4.4\nOA: 0.8\n"
print("recovered from a partial run:", fpl.parse_metrics(partial))

---

## 5. Convert your own layouts 🟢

`floorplanlab.convert` turns layout CSVs into the JSON schema the models expect. A sample layout
ships with the library.

In [ ]:
import tempfile
from pathlib import Path
from floorplanlab.convert import csv_to_json
from floorplanlab.examples import sample_layout

out = Path(tempfile.mkdtemp())          # csv_to_json writes a .json file; keep it out of the way
data = csv_to_json(str(sample_layout()), output_dir=str(out), show_plot=False)

print("components :", len(data["zone_types"]))
print("zone types :", data["zone_types"])
print("orientations:", data["orient"])

Zone types are `1–4` rooms, `15` entrance door, `16` window, `17` interior door — and the
order is canonical: rooms first, then interior doors, then windows, entrance door always last.

Orientation is 8-way (`1` East, counter-clockwise to `8` South-East). `0` means "no direction":
doors always get it, and so does a space sitting at the plan's centre.

### The environmental conditions

This is what makes the `ohdw-oesd` model *context-aware*. Each label is the orientation of the
single **best space** in the layout:

In [ ]:
print("AIF — orientation of the sunniest windowed space :", data["AIF"])
print("VL  — orientation of the space with the best view  :", data["VL"])
print("NN  — orientation of the quietest windowed space   :", data["NN"])

So instead of only *"this room faces south"*, the model can be told *"the sunniest room
should face this way, the quietest that way"*.

### Tolerances are configurable

In the original research notebook these were hardcoded globals. Now they are a validated config
object, so you can try a variation without editing source:

In [ ]:
from floorplanlab.convert import ConvertConfig

wider = csv_to_json(str(sample_layout()), output_dir=str(out), show_plot=False,
                    cfg=ConvertConfig(margin=60))

print("same components:", data["zone_types"] == wider["zone_types"])
print("different geometry:", data["edges"] != wider["edges"])

In [ ]:
# nonsense is rejected at construction, not 200 layouts later
try:
    ConvertConfig(snap_iou_threshold=1.5)
except ValueError as e:
    print("rejected:", e)

---

## 6. Build reproducible dataset splits 🟢

`make_splits` writes train / test / deploy sets, a provenance manifest, and the `list.txt`
files the sampling scripts require (forgetting those is a classic confusing failure).

In [ ]:
import json
from floorplanlab.convert import make_splits

# stand in for a folder of converted layouts of one target size
src = out / "converted"; src.mkdir()
for i in range(60):
    (src / f"layout_{i:03d}.json").write_text(json.dumps(data))

# pass {target_size: folder} — several sizes are split independently, so every
# split keeps the same mix of layout sizes
result = make_splits({6: src}, out / "dataset",
                     test_frac=0.10, deploy_frac=0.05, seed=123)
print(result)

The **deploy** split is the interesting one: it deliberately drops the geometry and keeps
only the conditions you would know *before* a layout exists.

In [ ]:
train_keys  = set(json.loads(next((out/"dataset"/"train").glob("*.json")).read_text()))
deploy_keys = set(json.loads(next((out/"dataset"/"deploy").glob("*.json")).read_text()))

print("train  keys:", sorted(train_keys))
print("deploy keys:", sorted(deploy_keys))
print("\ndropped for deploy:", sorted(train_keys - deploy_keys))

That is the difference between **reconstructing a plan the model has the answer to** and
**designing one from a brief**. Deployment generation uses only the second.

Splits are reproducible from the seed, and the manifest records exactly what went where:

In [ ]:
manifest = json.loads(result.manifest_path.read_text())
print({k: manifest[k] for k in ("seed", "test_frac", "deploy_frac")})
print("per target size:", manifest["per_target"])

---

## 7. Guardrails 🟢

A few things the library refuses to let you do quietly.

**Stages a model does not have.** Not every model can train or deploy — asking says so plainly
instead of failing deep inside a script.

In [ ]:
from floorplanlab.stages import Stage

for key in sorted(fpl.MODELS):
    can = [str(s) for s in Stage if fpl.get(key).supports(s)]
    print(f"{key:<12} {', '.join(can)}")

# asking a model for a stage it does not have says so plainly
try:
    fpl.get("hd-rplan").stage(Stage.TRAIN)
except KeyError as e:
    print("\nrejected:", e)

**Arguments a stage does not accept.** `argparse` would silently ignore these; the stage
definition catches them.

In [ ]:
from floorplanlab.stages import Stage

train_stage = fpl.get("ohdw-oesd").stage(Stage.TRAIN)
try:
    train_stage.build_args("oesd", target_set=7, num_samples=64)
except TypeError as e:
    print("rejected:", e)

**Ambiguous output files.** If an output folder ever holds results from more than one run,
pairing refuses to guess rather than silently plotting the wrong layout beside your input.

In [ ]:
from floorplanlab import pairing

amb = out / "ambiguous"; (amb / "graphs_gt").mkdir(parents=True); (amb / "pred").mkdir()
(amb / "graphs_gt" / "20.svg").write_text("<svg/>")
for name in ("20c_0_pred.svg", "20c_1_pred.svg"):     # two runs, merged
    (amb / "pred" / name).write_text("<svg/>")

print(pairing.build_pairs(amb / "graphs_gt", right_dir=amb / "pred").summary())

---

## Where next

- **[README](README.md)** — features, examples and requirements at a glance
- **[ARCHITECTURE.md](ARCHITECTURE.md)** — how the registry and stages fit together, module by module
- Adding a model should mean adding **one `ModelSpec`**. If it doesn't, that's a design bug
  worth reporting.

### The blue sections, when you are ready

You will need a CUDA GPU (Google Colab's T4 is fine — a 64-sample run takes about 13–16 minutes)
and a Google Drive folder containing the pre-trained weights, patched model sources and datasets.
Those are not redistributed with this library; see the README for where to get them.